<a href="https://colab.research.google.com/github/coopster-seclusion/GISC401_Cass_Group_Project/blob/main/GISC401_Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**GISC401 Cass Station Field Trip Group Assignment**

Task 1: Spatial Analysis and Processing of GPS Data

Group: Alan English, Steven Cooper, Amber Bowden, Tavake Tohi, and Zak Hynd

Data: GPS tracks collected by field trip student group = GPS_data / GPX files

**1: Connect to project GitHub repo to access data // install & import required libs**


In [2]:
# remove existing directory to ensure a fresh clone with latest updates
!rm -rf /content/GISC401_Cass_Group_Project

# clone the repo
!git clone https://github.com/coopster-seclusion/GISC401_Cass_Group_Project.git

# validate clone
import os
if os.path.exists('/content/GISC401_Cass_Project'):
    print("Contents of repo:", os.listdir('/content/GISC401_Cass_Group_Project'))
else:
    print("Repo folder not found.")

Cloning into 'GISC401_Cass_Group_Project'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 24 (delta 7), reused 21 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 413.30 KiB | 9.61 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Contents of repo: ['GPS_Data', 'GISC401_Westport.ipynb', 'README.md', '.git', 'Questions', 'tall git-filter-repo', 'GISC401_Task1.ipynb']


In [3]:
# folder paths - mapped to github repo
gps_path = "/content/GISC401_Cass_Project/GPS_Data"
questions_path = "/content/GISC401_Cass_Project/Questions"

# process gps files and sort
try:
    gpx_files = sorted([f for f in os.listdir(gps_path) if f.endswith('.gpx')])
    print(f"Found {len(gpx_files)} GPX files in GPS_Data (sorted): {gpx_files}")
except FileNotFoundError:
    print("GPS_Data folder not found in the specified path.")

# process questions files
try:
    q_files = os.listdir(questions_path)
    print(f"Found {len(q_files)} files in Questions: {q_files}")
except FileNotFoundError:
    print("Questions folder not found in the specified path.")

Found 9 GPX files in GPS_Data (sorted): ['A.gpx', 'B.gpx', 'C.gpx', 'D.gpx', 'E.gpx', 'F.gpx', 'G.gpx', 'H.gpx', 'I.gpx']
Found 3 files in Questions: ['Q1.md', 'Q2.md', 'about.txt']


In [4]:
# install project libs
import subprocess
import sys

libraries = ['gpxpy', 'movingpandas', 'stonesoup', 'geopandas', 'hvplot', 'geoviews']

print("Starting installation...")
for lib in libraries:
    subprocess.check_call([sys.executable, "-m", "pip", "install", lib, "-q"])
    print(f"{lib} installed successfully!")

print("\nAll required libraries installed successfully!")

Starting installation...
gpxpy installed successfully!
movingpandas installed successfully!
stonesoup installed successfully!
geopandas installed successfully!
hvplot installed successfully!
geoviews installed successfully!

All required libraries installed successfully!


In [5]:
# import libs
from pathlib import Path
from shapely.ops import transform
from datetime import datetime, timedelta
from IPython.display import Markdown, display

import gpxpy
import pandas as pd
import geopandas as gpd
import movingpandas as mpd
import pyproj
import hvplot.pandas
import geoviews as gv
import holoviews as hv
import numpy as np
import matplotlib.pyplot as plt
import statistics
import folium

In [6]:
""" utility module section using gpxpy and geopandas """

# parse gpx into geodataframe with datetimeindex function
def parse_gpx(path: str) -> gpd.GeoDataFrame:
    """
    Parse a GPX file into a tz-aware (UTC) GeoDataFrame indexed by timestamp.
    Handles both tz-aware and tz-naive datetimes from gpxpy.
    """
    with open(path) as f:
        gpx = gpxpy.parse(f)

    records = []
    for trk in gpx.tracks:
        for seg in trk.segments:
            for pt in seg.points:
                ts = pd.Timestamp(pt.time)
                # gpxpy returns tz-aware or tz-naive depending on the GPX file
                # normalise to UTC either way
                if ts.tzinfo is None:
                    ts = ts.tz_localize("UTC")
                else:
                    ts = ts.tz_convert("UTC")
                records.append({
                    "latitude":  pt.latitude,
                    "longitude": pt.longitude,
                    "elevation": pt.elevation,
                    "time":      ts,
                })

    df = pd.DataFrame(records).set_index("time").sort_index()
    return gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
        crs="EPSG:4326",
    )

def to_nz(ts: pd.Timestamp) -> str:
    """Convert a UTC timestamp (aware or naive) to NZ time string."""
    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")
    return ts.tz_convert(NZ_TZ).strftime("%Y-%m-%d %H:%M:%S")

def make_trajectory(gdf: gpd.GeoDataFrame, traj_id: str) -> mpd.Trajectory:
    """
    Wrap a GeoDataFrame in a movingpandas Trajectory.
    Timezone is stripped from the index prior to creation per movingpandas
    requirements — parse_gpx guarantees UTC so no information is lost.
    """
    gdf = gdf.copy()
    gdf.index = gdf.index.tz_localize(None)
    return mpd.Trajectory(gdf, traj_id=traj_id)

def compute_distance_km(gdf):
    # recalc distance by projecting to a metric crs nztm2000
    gdf_nztm = gdf.to_crs('EPSG:2193')
    # calc step distances between consecutive points
    dist = gdf_nztm.geometry.distance(gdf_nztm.geometry.shift())
    return dist.sum() / 1000

def recording_resolution(gdf):
    # calc median time difference between points in seconds
    diffs = gdf.index.to_series().diff().dt.total_seconds()
    return diffs.median()

def geodesic_distance_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Return geodesic distance in metres between two WGS84 lat/lon pairs."""
    geod = pyproj.Geod(ellps="WGS84")
    _, _, dist = geod.inv(lon1, lat1, lon2, lat2)
    return dist

def scale_size(series: pd.Series) -> pd.Series:
    """Min-max scale a duration series to a visual point-size range."""
    s_min, s_max = series.min(), series.max()
    if s_min == s_max:
        return pd.Series([(SIZE_MIN + SIZE_MAX) / 2] * len(series), index=series.index)
    return SIZE_MIN + (series - s_min) / (s_max - s_min) * (SIZE_MAX - SIZE_MIN)

# nz timezone
NZ_TZ = 'Pacific/Auckland'

# cass field station coords
CASS_LAT, CASS_LON = -43.034792, 171.759373

print("Notebook utility fuctions defined")

Notebook utility fuctions defined


In [7]:
# questions function to pull markdown answers and display word count

def display_question(filename):
    # reads file from the questions path, calculates word count, and displays it
    full_path = os.path.join(questions_path, filename)

    try:
        with open(full_path, 'r') as f:
            content = f.read()

        # word count: split by whitespace
        word_count = len(content.split())

        print(f"--- Source: {filename} | Word Count: {word_count} ---")
        display(Markdown(content))

    except FileNotFoundError:
        print(f"Error: The file {filename} was not found in {questions_path}.")

## **Q1: What is the integrated science of movement? <= 300 words**


In [8]:
# questions function to display the answer for Q1
display_question('Q1.md')

--- Source: Q1.md | Word Count: 315 ---


The **integrated science of movement** is an interdisciplinary field that synthesises methods and theory from ecology, geography, computer science, and statistics to study how and why entities move through space and time (Laube, 2014). Rather than treating movement as simple displacement from origin to destination, the field examines the full complexity of trajectories — the continuous paths that moving objects trace — and seeks to extract meaningful patterns such as preferred routes, stops, turning behaviour, speed profiles, and interactions between movers.

The theoretical backbone is **Computational Movement Analysis (CMA)**, which provides a formal framework for representing, querying, and mining movement data (Laube, 2014). CMA treats a trajectory as a time-ordered sequence of space–time positions and develops algorithms to characterise properties including speed, direction, sinuosity, clustering, and periodicity. These properties reveal phenomena as diverse as animal migration corridors, urban commuter flow, or — as in this assignment — the route and behaviour of field researchers during a data-collection campaign.

**Visual analytics** forms a complementary pillar, offering interactive and automated techniques for exploring high-dimensional movement datasets (Andrienko & Andrienko, 2013). By coupling algorithmic detection with human-guided exploration through linked maps, graphs, and filters, visual analytics allows analysts to discover unexpected movement phenomena that purely automated methods might overlook.

In practice the integrated science of movement pairs GPS trajectory data with geographic context layers — elevation models, land cover, road networks — to interpret not just *where* an entity went, but *why*. Stop detection algorithms, for instance, identify periods of minimal or no displacement that may correspond to rest, refuelling, or points of interest. The field's integrative character is its core strength: no single discipline could provide both the mathematical rigour of spatial algorithms and the domain interpretation required to turn raw coordinates into scientific insight.

### References

Andrienko, G., & Andrienko, N. (2013). *Visual analytics of movement*. Springer. https://doi.org/10.1007/978-3-642-37583-5

Laube, P. (2014). *Computational movement analysis*. Springer. https://doi.org/10.1007/978-3-319-10268-9


## **Q2: How does GPS time work? What timezone is the raw GPX data in? <= 100 words**


In [9]:
# questions function to display the answer for Q2
display_question('Q2.md')

--- Source: Q2.md | Word Count: 106 ---


GPS satellites carry atomic clocks synchronised to **UTC** (Coordinated Universal Time). Receivers derive time from satellite signals and output timestamps in UTC, identified in GPX files by a trailing `Z` suffix. GPS time and UTC differ by accumulated leap seconds (currently 18 s), but GPS receivers apply this correction automatically, so the logged timestamps are true UTC.

The raw `.gpx` files in this dataset are therefore in **UTC**. The embedded metadata (`<mytracks:timezone offset="780"/>`) confirms the device's local timezone as UTC + 780 min = **UTC+13 (NZDT)**, which applied on 2026-03-14 in New Zealand.

### Reference

National Institute of Standards and Technology. (2023). *GPS and UTC*. https://www.nist.gov/pml/time-and-frequency-division/time-realization/gps-utc


## **Q3: Convert timestamps to NZ time; produce summary table**


In [10]:
""" process gpx files to build q3 summary table"""

records = []

for filename in gpx_files:
    gdf = parse_gpx(str(Path(gps_path) / filename))
    records.append({
        "OG GPS file":   filename,
        "NZ start timestamp":  to_nz(gdf.index[0]),
        "NZ end timestamp":    to_nz(gdf.index[-1]),
        "Distance Covered (km)":       round(compute_distance_km(gdf), 2),
        "# of GPS point":     len(gdf),
        "Recording resolution": f"{int(recording_resolution(gdf))}s",
    })

summary_df = pd.DataFrame(records)
display(summary_df)

,OG GPS file,NZ start timestamp,NZ end timestamp,Distance Covered (km),# of GPS point,Recording resolution
0,A.gpx,2026-03-14 07:42:06,2026-03-14 10:48:18,120.19,80,120s
1,B.gpx,2026-03-14 07:42:07,2026-03-14 10:43:18,128.13,901,10s
2,C.gpx,2026-03-15 07:40:30,2026-03-15 10:40:30,99.95,7,1800s
3,D.gpx,2026-03-14 07:42:30,2026-03-14 10:47:32,132.88,167,61s
4,E.gpx,2026-03-15 07:37:40,2026-03-15 10:45:19,117.98,96,120s
5,F.gpx,2026-03-15 07:32:01,2026-03-15 10:49:05,126.96,3852,3s
6,G.gpx,2026-03-15 09:51:21,2026-03-15 10:04:53,0.51,164,5s
7,H.gpx,2026-03-15 07:41:31,2026-03-15 10:44:21,123.08,400,30s
8,I.gpx,2026-03-15 07:41:37,2026-03-15 10:41:37,104.90,13,900s


## **Q4: Maximum elevation across all GPS files**


In [11]:
"""calculate max elevation across all gps files"""
print("Q4: Max elevation calculated:")
print("=" * 45)

all_gdfs = {
    filename: parse_gpx(str(Path(gps_path) / filename))
    for filename in gpx_files
}

# find the file containing the global max elevation
max_file = max(
    all_gdfs,
    key=lambda f: all_gdfs[f]["elevation"].max()
)
gdf = all_gdfs[max_file]
peak = gdf.loc[gdf["elevation"].idxmax()]

max_elev = peak["elevation"]
max_lat  = peak["latitude"]
max_lon  = peak["longitude"]
max_time_nz = to_nz(peak.name)   # index is the UTC timestamp

max_elev_df = pd.DataFrame([{
    "File":       max_file,
    "Elevation":  max_elev,
    "Latitude":   max_lat,
    "Longitude":  max_lon,
    "Time (NZ)":  max_time_nz,
}])

print(f"Max elevation     : {max_elev:.1f} m")
print(f"Time (NZ)         : {max_time_nz}")
print(f"Location (WGS84)  : ({max_lat:.6f}, {max_lon:.6f})")
print(f"File              : {max_file}")

Q4: Max elevation calculated:
Max elevation     : 967.1 m
Time (NZ)         : 2026-03-15 09:26:22
Location (WGS84)  : (-43.296501, 171.740986)
File              : F.gpx


In [12]:
"""mapping function using hvplot and geoview - osm tile server is glitchy //
and often blocks non-browser request"""

# padding for zoomed out map rendering
PAD = 0.5

max_map = max_elev_df.hvplot.points(
    "Longitude", "Latitude",
    geo=True,
    tiles="CartoLight",
    color="Green",
    size=300,
    title=f"Q4: Max Elevation {max_elev:.1f} m | {max_time_nz} | {max_file}",
    hover_cols=["Elevation", "File"],
    frame_width=700,
    frame_height=500,
    xlim=(max_lon - PAD, max_lon + PAD),
    ylim=(max_lat - PAD, max_lat + PAD),
)
display(max_map)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]   (Elevation,File)

## **Q5: Stop detection per trajectory using a threshold of 120 sec and max diameter of 150 meters.**


In [13]:
# from pathlib import Path

MIN_DURATION = timedelta(seconds=120)
MAX_DIAMETER = 150  # metres

stop_records = []

print("Q5: Name of original GPX file and the number of stops detected:")
print("=" * 65)

for filename in gpx_files:
    full_path = Path(gps_path) / filename
    gdf = parse_gpx(str(full_path))

    if len(gdf) < 3:
        print(f"{full_path.name}: skipped — only {len(gdf)} point(s)")
        continue

    traj = make_trajectory(gdf, full_path.stem)
    detector = mpd.TrajectoryStopDetector(traj)
    stops = detector.get_stop_points(max_diameter=MAX_DIAMETER, min_duration=MIN_DURATION)

    for _, s in stops.iterrows():
        stop_records.append({
            "File":          full_path.name,
            "Start (NZ)":   to_nz(s["start_time"]),
            "End (NZ)":     to_nz(s["end_time"]),
            "Duration (s)": int(round(s["duration_s"])),
            "Latitude":     round(s.geometry.y, 5),
            "Longitude":    round(s.geometry.x, 5),
        })
    print(f"{full_path.name}: {len(stops)} stop(s) detected")

stops_df = pd.DataFrame(stop_records)
stops_df

Q5: Name of original GPX file and the number of stops detected:
A.gpx: 9 stop(s) detected
B.gpx: 8 stop(s) detected
C.gpx: 1 stop(s) detected
D.gpx: 9 stop(s) detected
E.gpx: 6 stop(s) detected
F.gpx: 12 stop(s) detected
G.gpx: 2 stop(s) detected
H.gpx: 8 stop(s) detected
I.gpx: 1 stop(s) detected


,File,Start (NZ),End (NZ),Duration (s),Latitude,Longitude
0,A.gpx,2026-03-14 08:32:17,2026-03-14 09:01:08,1730,-43.38974,172.02088
1,A.gpx,2026-03-14 09:39:09,2026-03-14 09:43:08,240,-43.19615,171.74162
2,A.gpx,2026-03-14 09:45:08,2026-03-14 09:47:08,120,-43.19653,171.74348
3,A.gpx,2026-03-14 09:49:08,2026-03-14 09:53:08,240,-43.19757,171.74417
4,A.gpx,2026-03-14 09:55:08,2026-03-14 10:01:08,360,-43.19624,171.74282
5,A.gpx,2026-03-14 10:03:08,2026-03-14 10:05:08,120,-43.19542,171.74304
6,A.gpx,2026-03-14 10:07:08,2026-03-14 10:13:09,360,-43.19635,171.74170
7,A.gpx,2026-03-14 10:37:44,2026-03-14 10:39:44,120,-43.03561,171.75706
8,A.gpx,2026-03-14 10:41:44,2026-03-14 10:48:18,393,-43.03465,171.75936
9,B.gpx,2026-03-14 08:30:52,2026-03-14 09:01:54,1863,-43.38974,172.02098


In [14]:
"""Map detected stops using hvplot — consistent with Q4 mapping style."""
PAD = 0.1  # tighter padding — stops are clustered along one route

stops_map = stops_df.hvplot.points(
    "Longitude", "Latitude",
    geo=True,
    tiles="CartoLight",
    color="File",
    size=200,
    title="Q5: Detected Stops per Trajectory (120s threshold, 150m diameter)",
    hover_cols=["File", "Start (NZ)", "End (NZ)", "Duration (s)"],
    frame_width=700,
    frame_height=500,
    xlim=(stops_df["Longitude"].min() - PAD, stops_df["Longitude"].max() + PAD),
    ylim=(stops_df["Latitude"].min() - PAD, stops_df["Latitude"].max() + PAD),
    legend=True,
)
display(stops_map)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]   (File,Start (NZ),End (NZ),Duration (s))

## **Q6: Excluding Cass Station, where was the longest stop? How was this calculated from the GPS data?.**


In [15]:
# Buffer radius around Cass Station for exclusion
CASS_RADIUS_M = 500

# Compute geodesic distance from each stop centroid to Cass Station
stops_df["Distance to Cass (m)"]  = stops_df.apply(
    lambda row: geodesic_distance_m(
        row["Latitude"], row["Longitude"], CASS_LAT, CASS_LON
    ),
    axis=1,
).round(0)
stops_df["Distance to Cass (km)"] = (stops_df["Distance to Cass (m)"] / 1000).round(2)

# Filter to stops outside Cass Station exclusion radius
outside_df = stops_df[stops_df["Distance to Cass (m)"] > CASS_RADIUS_M].copy()

# Identify the longest such stop
longest = outside_df.loc[outside_df["Duration (s)"].idxmax()]

print("Q6: Longest stop outside Cass Station")
print("=" * 40)
print(f"Duration        : {longest['Duration (s)']} s ({longest['Duration (s)'] / 60:.1f} min)")
print(f"Location        : ({longest['Latitude']:.5f}, {longest['Longitude']:.5f})")
print(f"Start (NZ)      : {longest['Start (NZ)']}")
print(f"End (NZ)        : {longest['End (NZ)']}")
print(f"File            : {longest['File']}")
print(f"Distance to Cass: {longest['Distance to Cass (m)']:.0f} m ({longest['Distance to Cass (km)']} km)")

Q6: Longest stop outside Cass Station
Duration        : 1920 s (32.0 min)
Location        : (-43.38948, 172.02034)
Start (NZ)      : 2026-03-15 08:28:54
End (NZ)        : 2026-03-15 09:00:54
File            : E.gpx
Distance to Cass: 44748 m (44.75 km)


In [16]:
"""Q6: Map longest stop outside Cass Station — circles scaled to stop duration."""
PAD = 0.15
SIZE_MIN, SIZE_MAX = 50, 500

# Build plot DataFrame
outside_plot_df = outside_df.copy()
outside_plot_df["Type"] = "Other stops"
outside_plot_df.loc[longest.name, "Type"] = "Longest stop"
outside_plot_df["Dist to Cass km"] = outside_plot_df["Distance to Cass (km)"]
outside_plot_df["point_size"]      = scale_size(outside_plot_df["Duration (s)"])

other_stops_df  = outside_plot_df[outside_plot_df["Type"] == "Other stops"]
longest_stop_df = outside_plot_df[outside_plot_df["Type"] == "Longest stop"]

hover_cols = ["File", "Start (NZ)", "End (NZ)", "Duration (s)", "Dist to Cass km"]

shared = dict(
    geo=True,
    frame_width=700,
    frame_height=500,
    xlim=(outside_plot_df["Longitude"].min() - PAD, outside_plot_df["Longitude"].max() + PAD),
    ylim=(outside_plot_df["Latitude"].min() - PAD, outside_plot_df["Latitude"].max() + PAD),
)

other_layer = other_stops_df.hvplot.points(
    "Longitude", "Latitude",
    tiles="CartoLight",
    color="yellow",
    size="point_size",
    label="Other stops",
    title="Q6: Longest Stop Outside Cass Station - Sheffield Pie Shop (circle size = stop duration)",
    hover_cols=hover_cols,
    **shared,
)

longest_layer = longest_stop_df.hvplot.points(
    "Longitude", "Latitude",
    color="red",
    size="point_size",
    marker="circle",
    label="Longest stop",
    hover_cols=hover_cols,
    **shared,
)

cass_df = pd.DataFrame([{
    "Longitude": CASS_LON,
    "Latitude":  CASS_LAT,
    "Label":     "Cass Station",
}])

cass_layer = cass_df.hvplot.points(
    "Longitude", "Latitude",
    color="blue",
    size=250,
    marker="triangle",
    label="Cass Station",
    hover_cols=["Label"],
    **shared,
)

display((other_layer * longest_layer * cass_layer).opts(legend_position="bottom_right"))

:Overlay
   .WMTS.I              :WMTS   [Longitude,Latitude]
   .Points.Other_stops  :Points   [Longitude,Latitude]   (point_size,File,Start (NZ),End (NZ),Duration (s),Dist to Cass km)
   .Points.Longest_stop :Points   [Longitude,Latitude]   (point_size,File,Start (NZ),End (NZ),Duration (s),Dist to Cass km)
   .Points.Cass_Station :Points   [Longitude,Latitude]   (Label)

## **Q7: GPS data quality assessment**


### Q7a: Is there variation of time delta? How did you verify this using the data?


In [17]:
"""Q7a: Analyse time delta variation across all GPS trajectories."""

print(f"{'File':<12} {'Points':>7} {'Min Δt (s)':>11} {'Max Δt (s)':>11} {'Mode Δt (s)':>12} {'Unique Δt':>10} {'Variation?':>11}")
print("-" * 80)

for filename in gpx_files:
    gdf = parse_gpx(str(Path(gps_path) / filename))
    times = gdf.index

    if len(times) < 2:
        print(f"{filename:<12} {'< 2 points — skipped':>60}")
        continue

    deltas    = [(times[i + 1] - times[i]).total_seconds() for i in range(len(times) - 1)]
    mode_dt   = statistics.mode(deltas)
    unique_dt = len(set(deltas))
    has_variation = "Yes" if unique_dt > 1 else "No"

    print(
        f"{filename:<12} {len(gdf):>7} {min(deltas):>11.1f} {max(deltas):>11.1f}"
        f" {mode_dt:>12.1f} {unique_dt:>10} {has_variation:>11}"
    )

File          Points  Min Δt (s)  Max Δt (s)  Mode Δt (s)  Unique Δt  Variation?
--------------------------------------------------------------------------------
A.gpx             80       120.0      1730.4        120.0         14         Yes
B.gpx            901         9.0      1723.3         10.0        242         Yes
C.gpx              7      1799.0      1801.0       1800.0          3         Yes
D.gpx            167        54.3       253.1         61.2        146         Yes
E.gpx             96        60.0       120.0        120.0          5         Yes
F.gpx           3852         2.7        30.0          3.0        539         Yes
G.gpx            164         1.0         5.9          5.0          5         Yes
H.gpx            400         1.0       405.9         30.0        233         Yes
I.gpx             13       900.0       900.9        900.0          5         Yes


### Q7b: Duplicated trajectories?


In [18]:
"""Q7b: Identify potentially duplicated trajectories by shared recording resolution."""
resolution_groups: dict[int, list[dict]] = {}

for filename in gpx_files:
    gdf = parse_gpx(str(Path(gps_path) / filename))
    res = int(recording_resolution(gdf))
    nz_date = gdf.index[0].tz_convert(NZ_TZ).strftime("%Y-%m-%d")
    resolution_groups.setdefault(res, []).append({
        "file": filename,
        "date": nz_date,
    })

print("Recording resolution → files sharing that resolution")
print("-" * 65)
for res_s, entries in sorted(resolution_groups.items()):
    flag = " ← POTENTIAL DUPLICATE" if len(entries) > 1 else ""
    files_dates = ", ".join(f"{e['file']} ({e['date']})" for e in entries)
    print(f"{res_s:>6}s : {files_dates}{flag}")

# Summarise duplicate groups with date comparison
print()
print("Interpretation")
print("-" * 65)
for res_s, entries in sorted(resolution_groups.items()):
    if len(entries) > 1:
        files     = [e["file"] for e in entries]
        dates     = [e["date"] for e in entries]
        same_day  = len(set(dates)) == 1
        day_note  = (
            "same calendar date — may be the same journey"
            if same_day
            else f"different calendar dates ({' vs '.join(dates)}) — separate days, same methodology"
        )
        print(f"  {', '.join(files)} share {res_s}s resolution → {day_note}")

Recording resolution → files sharing that resolution
-----------------------------------------------------------------
     3s : F.gpx (2026-03-15)
     5s : G.gpx (2026-03-15)
    10s : B.gpx (2026-03-14)
    30s : H.gpx (2026-03-15)
    61s : D.gpx (2026-03-14)
   120s : A.gpx (2026-03-14), E.gpx (2026-03-15) ← POTENTIAL DUPLICATE
   900s : I.gpx (2026-03-15)
  1800s : C.gpx (2026-03-15)

Interpretation
-----------------------------------------------------------------
  A.gpx, E.gpx share 120s resolution → different calendar dates (2026-03-14 vs 2026-03-15) — separate days, same methodology


### Q7c: How does tracking resolution affectg path accuracy?


In [19]:
"""Q7c: Visualise all collected trajectories by actual recording resolution."""
import folium

# Build per-file metadata
file_meta = {}
for filename in gpx_files:
    gdf = parse_gpx(str(Path(gps_path) / filename))
    res = int(recording_resolution(gdf))
    file_meta[filename] = {
        "gdf":          gdf,
        "resolution_s": res,
        "path_km":      round(compute_distance_km(gdf), 2),
    }

# Assign a colour per unique resolution
unique_res = sorted(set(m["resolution_s"] for m in file_meta.values()))
palette    = ["red", "blue", "green", "orange", "purple", "black", "cyan", "magenta"]
res_colour = {res: palette[i] for i, res in enumerate(unique_res)}

# Centre on mean of all points
all_lats = [pt for m in file_meta.values() for pt in m["gdf"]["latitude"]]
all_lons = [pt for m in file_meta.values() for pt in m["gdf"]["longitude"]]
centre   = [sum(all_lats) / len(all_lats), sum(all_lons) / len(all_lons)]

m = folium.Map(location=centre, zoom_start=13, tiles="CartoDB positron")

for filename, meta in file_meta.items():
    gdf    = meta["gdf"]
    res_s  = meta["resolution_s"]
    colour = res_colour[res_s]
    coords = list(zip(gdf["latitude"], gdf["longitude"]))

    folium.PolyLine(
        coords,
        color=colour,
        weight=2.5,
        opacity=0.8,
        tooltip=f"{filename} | {res_s}s | {len(gdf)} pts | {meta['path_km']} km",
    ).add_to(m)

# Legend
legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
            background:white; padding:10px; border-radius:6px;
            border:1px solid #ccc; font-size:13px; line-height:1.9;">
    <b>Q7c: Trajectories by Resolution</b><br>
"""
for res_s, colour in sorted(res_colour.items()):
    files_at_res = [f for f, m in file_meta.items() if m["resolution_s"] == res_s]
    legend_html += (
        f'<span style="color:{colour};">&#9644;</span> '
        f'{res_s}s — {", ".join(files_at_res)}<br>'
    )
legend_html += "</div>"

m.get_root().html.add_child(folium.Element(legend_html))
display(m)

# Summary table
res_summary = pd.DataFrame([
    {
        "File":           filename,
        "Resolution (s)": meta["resolution_s"],
        "Point count":    len(meta["gdf"]),
        "Path length (km)": meta["path_km"],
    }
    for filename, meta in sorted(file_meta.items(), key=lambda x: x[1]["resolution_s"])
])
display(res_summary)

,File,Resolution (s),Point count,Path length (km)
0,F.gpx,3,3852,126.96
1,G.gpx,5,164,0.51
2,B.gpx,10,901,128.13
3,H.gpx,30,400,123.08
4,D.gpx,61,167,132.88
5,A.gpx,120,80,120.19
6,E.gpx,120,96,117.98
7,I.gpx,900,13,104.90
8,C.gpx,1800,7,99.95
